## About

Explore the functionality of various tools that are available in the medea.
This helps with understanding how to use them for my work.

In [ ]:
import os
import json
import torch
import pandas as pd


ImportError: attempted relative import with no known parent package

In [14]:

base_url = "https://api.platform.opentargets.org/api/v4/graphql"

open_target_query_string = """
    query target($diseaseName: String!){
        search(queryString: $diseaseName){
        hits{
            id
            entity
            category
            name
            description
        }
    }
}
"""


In [2]:
def load_pinnacle_ppi(
    cell_type: str,
    embed_path: str = None, 
    weights_only=False
):
    """
    Load the PINNACLE PPI embeddings for a specific cell type from a specified path.
    
    Uses the same normalization and matching logic as celltype_avaliability_checker() to ensure consistency.
    Accepts multiple cell type formats and normalizes them automatically.
    
    Args:
        cell_type (str): The specific cell type to load embeddings for. Accepts multiple formats:
                        - Standardized: 'cd4_positive_alpha_beta_memory_t_cell'
                        - Raw: 'cd4-positive,_alpha-beta_memory_t_cell'
                        - User-friendly: 'CD4+ memory T cell'
                        All formats are normalized to the same internal representation.
        embed_path (str): Path to the PPI embeddings file. If None, uses MEDEADB_PATH environment variable.
        weights_only (bool): Whether to load weights only or the full state.
    
    Returns:
        dict: A dictionary of PPI embeddings for the specified cell type with gene name as key 
            and cell type-specific gene embedding as value (torch.Tensor). Returns empty dict if cell type not found.
    """
    # Set default embed_path if not provided
    if embed_path is None:
        embed_path = os.path.join(_get_medeadb_path(), 'pinnacle_embeds/ppi_embed_dict.pth')
    
    # Load the full PPI dictionary from the specified path
    ppi_dict = torch.load(embed_path, weights_only=weights_only)
    
    # Normalize cell type using same function as checker
    def _normalize(s):
        return s.replace(",", "").replace("-", "_").replace(" ", "_").replace("+", "_positive").replace("α", "alpha").replace("β", "beta").lower()
    
    def _format_display(s):
        """Format for clean, consistent display."""
        formatted = s.replace(",", "").replace("-", "_").replace(" ", "_")
        while "__" in formatted:
            formatted = formatted.replace("__", "_")
        return formatted.lower()
    
    formalized_cell_type = _normalize(cell_type)
    
    # First pass: exact match
    for cell_key in ppi_dict.keys():
        formalized_key = _normalize(cell_key)
        if formalized_key == formalized_cell_type:
            formatted_name = _format_display(cell_key)
            print(f"[load_pinnacle_ppi] ✓ Loaded {len(ppi_dict[cell_key])} genes for cell type: '{formatted_name}'", flush=True)
            return ppi_dict[cell_key]
        
    # Second pass: fuzzy matching with same logic as checker
    from thefuzz import fuzz
    best_match = None
    best_score = 0
    
    for cell_key in ppi_dict.keys():
        candidate_norm = _normalize(cell_key)
        
        # Compute similarity score using multiple strategies
        token_sort = fuzz.token_sort_ratio(formalized_cell_type, candidate_norm)
        token_set = fuzz.token_set_ratio(formalized_cell_type, candidate_norm)
        
        # Combined score
        score = token_sort * 0.7 + token_set * 0.3
        
        # Token overlap boost
        query_tokens = set(formalized_cell_type.split('_'))
        candidate_tokens = set(candidate_norm.split('_'))
        if query_tokens and candidate_tokens:
            intersection = len(query_tokens.intersection(candidate_tokens))
            union = len(query_tokens.union(candidate_tokens))
            jaccard = intersection / union if union > 0 else 0
            score += jaccard * 15
        
        if score > best_score:
            best_score = score
            best_match = cell_key
    
    # Return best match if score is reasonable
    if best_match and best_score >= 60:
        formatted_name = _format_display(best_match)
        print(f"[load_pinnacle_ppi] ⚠ Cell type '{cell_type}' matched to '{formatted_name}' (score: {best_score:.0f})", flush=True)
        print(f"[load_pinnacle_ppi] ✓ Loaded {len(ppi_dict[best_match])} genes for matched cell type", flush=True)
        return ppi_dict[best_match]
    
    # If no good match found, return empty dict
    print(f"[load_pinnacle_ppi] ✗ ERROR: Cell type '{cell_type}' not found in PINNACLE embeddings (best match score: {best_score:.0f}).", flush=True)
    print(f"[load_pinnacle_ppi] → Please use the celltype_avaliability_checker to find valid cell types.", flush=True)
    return {}


In [ ]:
## ppi_embed_dict returns gene as key and 128 dim embedding as vector

ppi_embed_dict = load_pinnacle_ppi(cell_type = "cd4_positive_alpha_beta_memory_t_cell", 
                                    embed_path = os.path.join("..","database","MedeaDB","pinnacle_embeds","ppi_embed_dict.pth"), 
                                    weights_only = False)

[load_pinnacle_ppi] ✓ Loaded 1067 genes for cell type: 'cd4_positive_alpha_beta_memory_t_cell'


In [12]:
list(ppi_embed_dict.items())[10][-1].shape

torch.Size([128])

## Check the open target API access

In [16]:

import requests
import re
def search_disease_open_target(disease_name):
    # Set variables object of arguments to be passed to endpoint
    relavent_entities = None
    variables = {"diseaseName": disease_name}
    for i in range(5):
        try:
            # Perform POST request and check status code of response
            r = requests.post(base_url, json={"query": open_target_query_string, "variables": variables})
            # Transform API response from JSON into Python dictionary and print in console
            api_response = json.loads(r.text)
            relavent_entities = api_response['data']['search']['hits']
        except Exception as e:
            print(f"[OpenTarget] Bad OpenTarget API response, retrying...")
            continue
        if relavent_entities is not None: break

    if relavent_entities is None:
        raise ValueError(f"[OpenTarget] No relevant entities found for {disease_name}")
    # Check the top 5 entities
    filtered_entities = [ e for e in relavent_entities if e['entity'] == 'disease']
    return filtered_entities

In [17]:
search_disease_open_target(disease_name="rheumatoid arthritis")

[{'id': 'EFO_0000685',
  'entity': 'disease',
  'category': ['musculoskeletal or connective tissue disease',
   'immune system disease'],
  'name': 'rheumatoid arthritis',
  'description': 'A chronic, systemic autoimmune disorder characterized by inflammation in the synovial membranes and articular surfaces. It manifests primarily as a symmetric, erosive polyarthritis that spares the axial skeleton and is typically associated with the presence in the serum of rheumatoid factor.'},
 {'id': 'HP_0001370',
  'entity': 'disease',
  'category': ['phenotype'],
  'name': 'Rheumatoid arthritis',
  'description': 'Inflammatory changes in the synovial membranes and articular structures with widespread fibrinoid degeneration of the collagen fibers in mesenchymal tissues, as well as atrophy and rarefaction of bony structures.'},
 {'id': 'EFO_0009460',
  'entity': 'disease',
  'category': ['musculoskeletal or connective tissue disease',
   'immune system disease'],
  'name': 'ACPA-negative rheumatoi

## Study the target protein retrival

In [1]:
from tool_space.action_functions import *

[Warning] ToolUniverse not available: No module named 'tooluniverse'
[Warning] Install tooluniverse to use Medea tools: pip install tooluniverse


In [2]:
target_set = load_disease_targets(disease_name="rheumatoid arthritis")

[load_disease_targets] Querying OpenTargets API for disease: rheumatoid arthritis (EFO_0000685)
[load_disease_targets] Found 4967 total target associations from API
[load_disease_targets] Fetching page 2 (retrieved 1872 targets so far)...
[load_disease_targets] Retrieved 1872 targets from OpenTargets API after filtering


In [3]:
target_set


{'XRCC4',
 'SEL1L',
 'NEK6',
 'CPEB4',
 'KPNA7',
 'MT-ND2',
 'KRTCAP2',
 'TUBB8',
 'GRB10',
 'SMTNL2',
 'MEN1',
 'DENND3',
 'ALOX5AP',
 'DPF3',
 'ADAM12',
 'PPAT',
 'PHC1',
 'CHRNG',
 'LPIN1',
 'SRSF4',
 'CMTR1',
 'IRF5',
 'DLD',
 'IL4R',
 'PDE8B',
 'RNFT1',
 'AHNAK2',
 'NXPE4',
 'SETD1A',
 'FCRL3',
 'INS',
 'FHL3',
 'CFAP119',
 'TAF11',
 'TMEM33',
 'CD80',
 'ALCAM',
 'GPR137',
 'PAQR5',
 'LIN7C',
 'LGALS9',
 'USP8',
 'MYRF',
 'ATG10',
 'GLIS3',
 'NARF',
 'FRAS1',
 'MEGF8',
 'IMMP2L',
 'GPSM1',
 'ELL',
 'GATA3',
 'KIF21B',
 'ALG10',
 'ABL1',
 'IL19',
 'METTL15',
 'IL23A',
 'NPAS2',
 'PSD4',
 'ENSG00000292259',
 'SELE',
 'TNFSF12',
 'POLR2B',
 'FGD2',
 'ACTR1B',
 'ELAPOR2',
 'RAB5B',
 'SCN8A',
 'SBF2',
 'NDNF',
 'ANKRD27',
 'ZIC4',
 'VWA8',
 'GNRHR',
 'TMC1',
 'SCUBE3',
 'TMEM151B',
 'UCHL3',
 'C10orf90',
 'DYDC2',
 'TRMT112',
 'TM6SF2',
 'UTP11',
 'MBL2',
 'PGAP3',
 'PRR14',
 'CAMTA1',
 'RPS23',
 'PTPN2',
 'ERICH1',
 'ZAP70',
 'XPR1',
 'PRDM5',
 'VGLL4',
 'IL6',
 'IGSF3',
 'FOXF2',
 'M

In [5]:
## Cross check the target set from OT with the evaluation datasets

ra_42_df = pd.read_csv(os.path.join("..","evaluation","targetID","source","targetid-ra-42.csv"))
ra_42_df.head()
celltype_to_y = ra_42_df.groupby('celltype')['y'].apply(list).to_dict()
celltype_to_y

{'classical monocyte': ['IL1B',
  'ADAM17',
  'AHR',
  'HLA-E',
  'HLA-DRA',
  'HIF1A',
  'TLE3',
  'RPLP1',
  'MTPAP',
  'IFNGR2',
  'ZEB2',
  'NFKBIA',
  'CDC42EP3',
  'ZEB2',
  'MTPAP',
  'CDC42EP3',
  'RPLP1',
  'IL1B',
  'NFKBIA',
  'TLE3'],
 'effector memory cd4 positive alpha beta t cell': ['CORO1A',
  'CD2',
  'HLA-C',
  'HLA-A',
  'RPL3',
  'CD52',
  'LTB',
  'CD3E',
  'JAK3',
  'RPLP1',
  'MT-ND3',
  'GPSM3',
  'S100A10',
  'RPL21',
  'PPIA',
  'IL6ST',
  'COX4I1',
  'PRDM1',
  'DDX6',
  'TNFAIP3'],
 'myeloid dendritic cell': ['RPS26',
  'COX4I1',
  'CD52',
  'HLA-E',
  'RPL21',
  'RPL3',
  'RPLP1',
  'LTB',
  'ACTB',
  'RPS26',
  'ACTB',
  'COX4I1',
  'RPL3',
  'CD52',
  'HLA-E',
  'RPL21',
  'LTB',
  'RPLP1',
  'RPLP1',
  'RPS26'],
 'naive b cell': ['MT-ND3',
  'MS4A1',
  'FOS',
  'NFKBIA',
  'IRF1',
  'YPEL5',
  'RPS26',
  'NFKBIA',
  'IRF1',
  'YPEL5',
  'MS4A1',
  'FOS',
  'RPS26',
  'MT-ND3',
  'IRF1',
  'MS4A1',
  'RPS26',
  'YPEL5',
  'MT-ND3',
  'NFKBIA'],
 'naive th

In [6]:
for celltype, targets in celltype_to_y.items():
    print(f"Cell type: {celltype}")
    overlap = set(targets) & target_set
    percentage_overlap = (len(overlap) / len(targets)) * 100 if targets else 0
    print(f"Percentage of overlap with target_set: {percentage_overlap:.2f}%")

Cell type: classical monocyte
Percentage of overlap with target_set: 40.00%
Cell type: effector memory cd4 positive alpha beta t cell
Percentage of overlap with target_set: 70.00%
Cell type: myeloid dendritic cell
Percentage of overlap with target_set: 25.00%
Cell type: naive b cell
Percentage of overlap with target_set: 30.00%
Cell type: naive thymus derived cd4 positive alpha beta t cell
Percentage of overlap with target_set: 55.00%
Cell type: naive thymus derived cd8 positive alpha beta t cell
Percentage of overlap with target_set: 50.00%
Cell type: natural killer cell
Percentage of overlap with target_set: 15.00%


## Next: For RA, what are the celltypes that can be targetted. is there any tool that talks about this?

User has to provide the celltype via the input query

In [1]:
from tool_space.open_alex import *

[Warning] ToolUniverse not available: No module named 'tooluniverse'
[Warning] Install tooluniverse to use Medea tools: pip install tooluniverse


In [12]:
import sys
import dotenv
dotenv.load_dotenv('.env')
#sys.path.append(os.path.dirname(os.path.dirname(os.path.abspath(__file__))))
#from tool_space.open_alex import search_openalex_papers, paper_search_from_openalex

# Test new semantic scholar compatible function
print("=== Testing search_openalex_papers (semantic scholar format) ===")
papers, keywords = search_openalex_papers("MCF7 Breast Cancer", max_results=3, open_access=True)
print(f"Keywords used: {keywords}")
print(f"Number of papers: {len(papers)}")
if papers:
    print(f"First paper format:")
    for key, value in papers[0].items():
        print(f"  {key}: {value}")

print("\n=== Testing paper_search_from_openalex (legacy format) ===")
result = paper_search_from_openalex("MCF7 Breast Cancer", max_results=2)
if isinstance(result, list) and result:
    print(f"Number of papers: {len(result)}")
    print(f"First paper keys: {list(result[0].keys())}")
else:
    print(f"Result: {result}")

=== Testing search_openalex_papers (semantic scholar format) ===
[INFO] Loading FlagEmbedding library (may download reranker models)...
[SEARCH_OPENALEX] WARNING: Keyword generation failed (No module named 'FlagEmbedding'), using question directly
[SEARCH_OPENALEX] DEBUG: Processing keyword 1/1: 'MCF7 Breast Cancer'
[OpenAlex] Retrieved 3 papers for keywords: 'MCF7 Breast Cancer'
[SEARCH_OPENALEX] SUCCESS: Found 3 papers for keyword: 'MCF7 Breast Cancer'
[SEARCH_OPENALEX] SUMMARY: 1/1 queries successful, 3 unique papers found
Keywords used: ['MCF7 Breast Cancer']
Number of papers: 3
First paper format:
  semantic_scholar_id: https://openalex.org/W2276797747
  type: openalex_abstract
  year: 2016
  authors: [{'authorId': 'https://openalex.org/A5022128827', 'name': 'Jhih-Pu Syu'}, {'authorId': 'https://openalex.org/A5066099267', 'name': 'Jen‐Tsan Chi'}, {'authorId': 'https://openalex.org/A5022430655', 'name': 'Hsiu‐Ni Kung'}]
  title: Nrf2 is the key to chemotherapy resistance in MCF7 br

In [14]:
keywords

['MCF7 Breast Cancer']

In [13]:
papers

[{'semantic_scholar_id': 'https://openalex.org/W2276797747',
  'type': 'openalex_abstract',
  'year': 2016,
  'authors': [{'authorId': 'https://openalex.org/A5022128827',
    'name': 'Jhih-Pu Syu'},
   {'authorId': 'https://openalex.org/A5066099267', 'name': 'Jen‐Tsan Chi'},
   {'authorId': 'https://openalex.org/A5022430655', 'name': 'Hsiu‐Ni Kung'}],
  'title': 'Nrf2 is the key to chemotherapy resistance in MCF7 breast cancer cells under hypoxia',
  'text': 'Hypoxia leads to reactive oxygen species (ROS) imbalance, which is proposed to associate with drug resistance and oncogenesis. Inhibition of enzymes of antioxidant balancing system in tumor cells was shown to reduce chemoresistance under hypoxia. However, the underlying mechanism remains unknown. The key regulator of antioxidant balancing system is nuclear factor erythroid 2-related factor 2 (NFE2L2, Nrf2). In this study, we showed that hypoxia induced ROS production and increased the Nrf2 activity. Nrf2 activation increased level